# Attention-variant report

Upload or locate `attention_variants_artifacts.zip`. This notebook performs no training or inference and does not require the dataset.

## Step 1 — Load and validate the artifact


In [1]:
import csv,io,json,zipfile,sys,collections
from pathlib import Path
from IPython.display import display,Markdown
NAME="attention_variants_artifacts.zip"
paths=[Path(NAME),Path("/content")/NAME,Path("/content/drive/MyDrive/CITS4012_A2_attention_variants")/NAME]
archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive; drive.mount("/content/drive"); archive=next((p for p in paths if p.exists()),None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    up=files.upload(); assert NAME in up; archive=Path("/content")/NAME; archive.write_bytes(up[NAME])
assert archive is not None
zf=zipfile.ZipFile(archive); assert zf.testzip() is None
summary=json.loads(zf.read("attention_variant_results/attention_summary.json")); print("Loaded",archive,"— no training or inference")


Loaded attention_variants_artifacts.zip — no training or inference


## Step 2 — Compare attention variants


In [2]:
labels={"AttnNone":"No self-attention","CurrentGated":"Current gated","ResidualAttn":"Ungated residual","QuestionGated":"Question-conditioned gated"}; order=list(labels); ms=("bleu1","meteor","rougeL","em","f1")
lines=["| Model | Params | BLEU-1 | METEOR | ROUGE-L | EM | F1 | ΔROUGE-L vs current |","|---|---:|---:|---:|---:|---:|---:|---:|"]
for n in order:
    v=summary["models"][n]; delta=v["rougeL"]-summary["models"]["CurrentGated"]["rougeL"]
    lines.append(f"| {labels[n]} | {v['params']:,} | "+" | ".join(f"{v[m]:.2f}" for m in ms)+f" | {delta:+.2f} |")
display(Markdown("## Controlled attention results\n\n"+"\n".join(lines)))


## Controlled attention results

| Model | Params | BLEU-1 | METEOR | ROUGE-L | EM | F1 | ΔROUGE-L vs current |
|---|---:|---:|---:|---:|---:|---:|---:|
| No self-attention | 3,699,753 | 37.62 | 32.48 | 42.63 | 26.67 | 42.70 | -1.51 |
| Current gated | 4,094,761 | 39.06 | 33.69 | 44.14 | 27.32 | 44.22 | +0.00 |
| Ungated residual | 3,963,433 | 38.88 | 32.97 | 43.98 | 28.35 | 44.17 | -0.16 |
| Question-conditioned gated | 4,160,553 | 39.36 | 33.47 | 44.36 | 28.44 | 44.42 | +0.22 |

## Step 3 — Run paired bootstrap analysis


In [3]:
import numpy as np
rows=list(csv.DictReader(io.StringIO(zf.read("attention_variant_results/paired_deltas.csv").decode()))); rng=np.random.default_rng(4012)
lines=["| Variant vs current | Metric | Mean Δ | 95% paired bootstrap CI | P(Δ≤0) |","|---|---|---:|---:|---:|"]
for model in ("AttnNone","ResidualAttn","QuestionGated"):
    subset=[r for r in rows if r["model"]==model]
    for metric in ("rougeL","f1","em"):
        d=np.array([float(r[metric]) for r in subset]); means=np.array([d[rng.integers(0,len(d),len(d))].mean() for _ in range(10000)])*100
        lo,hi=np.quantile(means,[.025,.975]); lines.append(f"| {labels[model]} | {metric} | {d.mean()*100:+.2f} | [{lo:+.2f}, {hi:+.2f}] | {(means<=0).mean():.3f} |")
display(Markdown("## Paired validation comparison\n\n"+"\n".join(lines)))


## Paired validation comparison

| Variant vs current | Metric | Mean Δ | 95% paired bootstrap CI | P(Δ≤0) |
|---|---|---:|---:|---:|
| No self-attention | rougeL | -1.51 | [-3.16, +0.14] | 0.964 |
| No self-attention | f1 | -1.53 | [-3.19, +0.09] | 0.967 |
| No self-attention | em | -0.65 | [-2.51, +1.21] | 0.773 |
| Ungated residual | rougeL | -0.16 | [-1.85, +1.53] | 0.571 |
| Ungated residual | f1 | -0.06 | [-1.79, +1.69] | 0.533 |
| Ungated residual | em | +1.02 | [-0.84, +2.88] | 0.154 |
| Question-conditioned gated | rougeL | +0.22 | [-1.34, +1.72] | 0.391 |
| Question-conditioned gated | f1 | +0.19 | [-1.37, +1.71] | 0.401 |
| Question-conditioned gated | em | +1.12 | [-0.46, +2.70] | 0.097 |

## Step 4 — Inspect question-type slices


In [4]:
lines=["| Variant | Question type | N | ΔROUGE-L |","|---|---|---:|---:|"]
for model in ("AttnNone","ResidualAttn","QuestionGated"):
    subset=[r for r in rows if r["model"]==model]
    groups=collections.defaultdict(list)
    for r in subset: groups[r["question_type"]].append(float(r["rougeL"])*100)
    for qt in sorted(groups): lines.append(f"| {labels[model]} | {qt} | {len(groups[qt])} | {sum(groups[qt])/len(groups[qt]):+.2f} |")
display(Markdown("## ROUGE-L effect by question type\n\n"+"\n".join(lines)+"\n\nSlice results are exploratory; small groups should not drive selection."))


## ROUGE-L effect by question type

| Variant | Question type | N | ΔROUGE-L |
|---|---|---:|---:|
| No self-attention | how | 85 | -2.56 |
| No self-attention | other | 95 | -4.97 |
| No self-attention | what | 462 | -1.28 |
| No self-attention | when | 24 | +6.79 |
| No self-attention | where | 75 | -5.43 |
| No self-attention | which | 22 | +1.53 |
| No self-attention | who | 290 | -0.03 |
| No self-attention | why | 23 | -5.33 |
| Ungated residual | how | 85 | +5.10 |
| Ungated residual | other | 95 | -0.43 |
| Ungated residual | what | 462 | +0.12 |
| Ungated residual | when | 24 | +2.46 |
| Ungated residual | where | 75 | -8.33 |
| Ungated residual | which | 22 | +2.71 |
| Ungated residual | who | 290 | -0.01 |
| Ungated residual | why | 23 | -4.70 |
| Question-conditioned gated | how | 85 | -0.81 |
| Question-conditioned gated | other | 95 | -2.94 |
| Question-conditioned gated | what | 462 | +1.13 |
| Question-conditioned gated | when | 24 | -4.89 |
| Question-conditioned gated | where | 75 | -4.31 |
| Question-conditioned gated | which | 22 | -0.82 |
| Question-conditioned gated | who | 290 | +2.30 |
| Question-conditioned gated | why | 23 | -6.56 |

Slice results are exploratory; small groups should not drive selection.

## Step 5 — Write the conclusion


In [5]:
current=summary["models"]["CurrentGated"]; candidates=[n for n in summary["models"] if n!="CurrentGated"]
best=max(candidates,key=lambda n:summary["models"][n]["rougeL"]); gain=summary["models"][best]["rougeL"]-current["rougeL"]
text=f"""# Attention-variant experiment

The strongest challenger was **{labels[best]}**, with validation ROUGE-L {summary['models'][best]['rougeL']:.2f}
({gain:+.2f} versus the current gated attention). This is a one-seed screening result. Only a challenger
that clearly exceeds the current model should proceed to a matched three-seed confirmation.
The locked test set was not accessed.
"""
Path("attention_variants_report.md").write_text(text,encoding="utf-8"); display(Markdown("## Report-ready conclusion\n\n"+text)); zf.close()


## Report-ready conclusion

# Attention-variant experiment

The strongest challenger was **Question-conditioned gated**, with validation ROUGE-L 44.36
(+0.22 versus the current gated attention). This is a one-seed screening result. Only a challenger
that clearly exceeds the current model should proceed to a matched three-seed confirmation.
The locked test set was not accessed.
